In [8]:
import time
import itertools
from KeyRecoveryScheme import h, Cipher, p
from Crypto.Cipher import AES
from Crypto.Util.Padding import unpad
from leak import leak

def lagrange_polynomial(i, x_lst, gf): # The polynomial L_i(x) according to the paper
    R.<X> = gf['X']
    result = 1
    for j in range(len(x_lst)):
        if (j == i):
            continue
        X_term = (X - x_lst[j]) / (x_lst[i] - x_lst[j])
        result *= X_term
    return result

def get_pts(lock, possible_answers, m, n, p): # enumerates possibilities at each x coordinate, based on possible values of share at each coordinate
    s_i = [int(s_ij) for s_ij in lock.decode().split(",")]
    pts = []
    assert(len(s_i) == n)
    for x in range(0, n):
        assert(len(possible_answers[x]) == m)
        s_ij = s_i[x]
        cand = []
        for i in range(m):
            cip = Cipher((x + 1, possible_answers[x][i]), p)
            dec = cip.decrypt(s_ij)
            cand.append(dec)
        pts.append((x + 1, cand))
    return pts

def precompute_y_times_L(pts, m, n, gf):
    (x_lst, y_cand) = zip(*pts)
    for cand in y_cand:
        assert(len(cand) == m)
    assert(len(x_lst) == len(y_cand))
    assert(n == len(x_lst))
    y_times_L = []
    for i in range(n):
        y_times_L_i = []
        for j in range(m):
            y_times_L_i.append(y_cand[i][j] * lagrange_polynomial(i, x_lst, gf))
        y_times_L.append(y_times_L_i)
    return y_times_L

def check_pair(p1, p2, k, gf): # check if pair of polynomials solve the problem
    R.<X> = gf['X']
    poly_diff = p1 - p2
    return (poly_diff.degree(X) <= k)

def poly_key(poly, k, n):
    poly_coeff = poly.coefficients(sparse=True)
    poly_coeff.extend([0] * (n - len(poly_coeff)))
    return poly_coeff[k + 1:]
    

def key_recovery_attack(lock, possible_answers, k, m, n, p):
    begin_time = time.time()
    pts = get_pts(lock, possible_answers, m, n, p)
    get_pts_time = time.time()
    print(f"Generated set of candidate y values for each x value ({get_pts_time - begin_time} seconds)")
    gf = GF(p)
    y_times_L = precompute_y_times_L(pts, m, n, gf)
    precompute_time = time.time()
    print(f"Precomputed lookup table for y_ij * L_i ({precompute_time - get_pts_time} seconds)")
    middle = len(y_times_L) // 2
    y_times_L_left = y_times_L[:middle]
    y_times_L_right = y_times_L[middle:]
    U_lst = [sum(poly) for poly in itertools.product(*y_times_L_left)]
    left_half_time = time.time()
    print(f"Possible left halves of interpolated polynomial sum computed ({left_half_time - precompute_time} seconds)")
    V_lst = [-1 * sum(poly) for poly in itertools.product(*y_times_L_right)]
    right_half_time = time.time()
    print(f"Possible negative right halves of interpolated polynomial sum computed ({right_half_time - left_half_time} seconds)")
    UV_merge = list(zip(U_lst, ["U"] * len(U_lst))) + list(zip(V_lst, ["V"] * len(V_lst)))
    UV_merge.sort(key=lambda x: poly_key(x[0], k, n))
    merge_time = time.time()
    print(f"Merged + sorted left and right halves ({merge_time - right_half_time} seconds)")
    for i in range(len(UV_merge) - 1):
        if (check_pair(UV_merge[i][0], UV_merge[i + 1][0], k, gf)):
            if (UV_merge[i][1] == "U" and UV_merge[i + 1][1] == "V"):
                poly = UV_merge[i][0] - UV_merge[i + 1][0]
            elif (UV_merge[i][1] == "V" and UV_merge[i + 1][1] == "U"):
                poly = UV_merge[i + 1][0] - UV_merge[i][0]
            else:
                continue
            match_time = time.time()
            print(f"Match found ({match_time - merge_time} seconds)")
            key = h(poly.coefficients(sparse=False)[0])
            print(f"key = {str(key)}")
            end_time = time.time()
            print(f"Total time of full attack: {end_time - begin_time} seconds")
            return key
    return None

In [6]:
possible_answers = leak
possible_answers

[['thomas',
  'miku',
  'harris',
  'baltimore',
  'hipple',
  'ayer',
  'ashe',
  'arming',
  'acre',
  'beckinham'],
 ['russell',
  'tom',
  'jerry',
  'ashley',
  'johnny',
  'benjamin',
  'fred',
  'gerry',
  'dumbo',
  'sesame'],
 ['san diego',
  'san jose',
  'sacramento',
  'san francisco',
  'new york city',
  'los angeles',
  'san bernandino',
  'trenton',
  'detroit',
  'columbus'],
 ['birdemic',
  'fateful findings',
  'the last airbender',
  'troll 2',
  'batman and robin',
  'fantastic four',
  'disaster movie',
  'manos',
  'catwoman',
  'secrets of dumbledore'],
 ['decision to leave',
  'oldboy',
  'memories of murder',
  'blade runner 2049',
  'puss in boots 2',
  'into the spider-verse',
  'blade runner',
  'mother',
  'parasite',
  'portrait of a lady on fire'],
 ['barcarolle',
  'american boy',
  'gimme gimme gimme',
  'naatu naatu',
  'smells like teen spirit',
  'something in the way',
  'party in the usa',
  'thrift shop',
  'not afraid',
  'levels'],
 ['call me m

In [9]:
lock = b'96772037615421034685797968900131327620,257995627570530992773640300250845929452,26560812263632337182013022569918906594,247192463880565299369883559093530349763,50057379004943689814052628144341770814,112233149607163743883531854115617268567,157064968171250910631970409603306108841,109239524680706367024485951974070578596,133819439707276950877568241347374334989,45433678570070248450191585464154736162'
key = key_recovery_attack(lock=lock, possible_answers=possible_answers, k=7, m=10, n=10, p=p)

Generated set of candidate y values for each x value (0.011245012283325195 seconds)
Precomputed lookup table for y_ij * L_i (0.024571895599365234 seconds)
Possible left halves of interpolated polynomial sum computed (0.8362529277801514 seconds)
Possible negative right halves of interpolated polynomial sum computed (2.032890796661377 seconds)
Merged + sorted left and right halves (7.078508615493774 seconds)
Match found (2.034442663192749 seconds)
key = b'>VQG\x0c\xd0\xfe\xd0SwC\xde\xa6](R'
Total time of full attack: 12.018051624298096 seconds


In [12]:
flag_aes = b'\xe0\xcbE\xd4!7\xe1\xdf\x17#C\xb1\x92\xa7\x92\xdc\xdd\x93\x84V\x86\r\x7f\x80\xf1\xaf\xd0Y\xc4\x8e{\xbe\xcbD\xc1\xad\x84\xcd3\x1e-|\x01,\x92U\xdc\xf6' # flag encrypted in AES-ECB mode (padded with PKCS7), using the key from KeyRecoveryScheme
aes_cipher = AES.new(key, AES.MODE_ECB)
unpad(aes_cipher.decrypt(flag_aes), 16)

b'SDCTF{m33t_m3_in_7h3_midd13_a9c527}'